# Environment

In [1]:
! pip install pandas


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
!pip install --upgrade torch transformers accelerate pillow


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
! pip install huggingface_hub


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


## Import Library

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import pandas as pd
import json

# Load Model

In [5]:
! nvidia-smi

Tue May 27 14:24:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.57.01              Driver Version: 565.57.01      CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:D1:00.0 Off |                    0 |
|  0%   35C    P8             23W /  300W |       1MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# token = "HF_TOKEN_REDACTED"

In [7]:
from huggingface_hub import login
import os

token = "HF_TOKEN_REDACTED"
os.environ['HF_TOKEN'] = token

login(token=token)
print("Logged in to Hugging Face!")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face!


In [8]:
"""from huggingface_hub import login
import os

# Get HF token injected from RunPod secret
hf_token = os.getenv('HF_TOKEN')

# Login
login(token=hf_token)

print("Hugging Face login successful via RunPod Secret!")"""

'from huggingface_hub import login\nimport os\n\n# Get HF token injected from RunPod secret\nhf_token = os.getenv(\'HF_TOKEN\')\n\n# Login\nlogin(token=hf_token)\n\nprint("Hugging Face login successful via RunPod Secret!")'

## Full Model

In [9]:
from transformers import AutoProcessor, LlavaForConditionalGeneration         
import torch, os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

model_id = "llava-hf/llava-1.5-7b-hf"

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",                  
    low_cpu_mem_usage=True,
)
# limit visual tokens to keep the vision encoder light
min_pixels = 224 * 28 * 28              
max_pixels = 768 * 28 * 28              
processor = AutoProcessor.from_pretrained(
    model_id, min_pixels=min_pixels, max_pixels=max_pixels
)

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


### Testing Response with Images

In [ ]:
# multiple images
from PIL import Image
import requests
import torch
import gc

# Helper to resize images for VRAM optimization
def resize_images(image_list, size=(384, 384)):
    return [img.resize(size, Image.BILINEAR) for img in image_list]

# Load and resize images
image_urls = [
    "https://media.istockphoto.com/id/1018245126/photo/siberian-tiger-portrait.jpg?s=612x612&w=0&k=20&c=ONrnG-JrcVuqdtIjfr4Pj85ZswUPt8wD-VbJOnWYPw0=",
    "https://images.pexels.com/photos/458991/pexels-photo-458991.jpeg?cs=srgb&dl=pexels-pixabay-458991.jpg&fm=jpg"
]
original_images = [Image.open(requests.get(url, stream=True).raw).convert("RGB") for url in image_urls]
images = resize_images(original_images, size=(384, 384))


question = "What animal is shown in each image?"

messages = [
    {
        "role": "user",
        "content": [{"type": "image", "image": img} for img in images] + [{"type": "text", "text": question}]
    }
]
print(messages)
# Generate prompt with Qwen-style chat template
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)

# Debug info
print("\n---- DEBUG INFO ----")
print("Generated Prompt:\n")
print(input_text)
print("\nNumber of <|vision_start|> tokens:", input_text.count("<|vision_start|>"))
print("Number of images passed:", len(images))
print("---------------------\n")

# Tokenize inputs
inputs = processor(
    text=[input_text],
    images=images,
    return_tensors="pt"
).to(model.device)

# Clear memory before generation
torch.cuda.empty_cache()
gc.collect()

# Generate response (deterministic)
with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
    generate_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

# Decode only model's output, skipping the prompt
prompt_len = inputs["input_ids"].shape[1]
reply_ids = generate_ids[:, prompt_len:]
output_text = processor.tokenizer.decode(reply_ids[0], skip_special_tokens=True)

# Output
print("\n# Response:\n")
print(output_text)


# Load Datasets

In [10]:
import pandas as pd
cloud=pd.read_excel("../../Dataset/cloud/split_data/cloud_test.xlsx")
device=pd.read_excel("../../Dataset/device/split_data/device_test.xlsx")

## Cloud

In [11]:
cloud.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Some recently changed resources are not yet pu...,\n \nEvery once in a while i ge...,"['java', 'eclipse', 'google-app-engine', 'java...",https://stackoverflow.com/questions/49711724/s...,4,2018-04-07 20:34:12Z,1,"[""@code511788465541441, if you consider Chanse...",[{'body': '\n \nIf that happens...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\n \nOur pipeline indicates suc...,"['azure-web-app-service', 'azure-deployment', ...",https://stackoverflow.com/questions/65255839/f...,7,2020-12-11 17:20:22Z,2,"[""thanks for the hint. i am going to verify th...","[{'body': ""\nI am almost certain that the pack...",\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


## Processed

In [12]:
cloud=cloud[['title','body','selected_answer','images']]

In [13]:
cloud.head(2)

,title,body,selected_answer,images
0,Some recently changed resources are not yet pu...,\n \nEvery once in a while i ge...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\n \nOur pipeline indicates suc...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


In [14]:
cloud['context'] = "title: " + cloud['title'].astype(str) + "\nbody: " + cloud['body'].astype(str).str.strip()

In [15]:
cloud_processed=pd.DataFrame(
    {
        'context':cloud['context'],
        'gt':cloud['selected_answer'],
        'img':cloud['images']
    }
)

In [16]:
cloud_processed.head(2)

,context,gt,img
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


## Device

In [17]:
device.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,What is the purpose of the INPUT chain in the ...,\n \nWhat is the purpose of the...,"['linux', 'iptables', 'linux', 'iptables']",https://serverfault.com/questions/993878/what-...,1,2019-12-01 00:23:16Z,1,"['OK, also please add this to your answer so o...",[{'body': '\nIf some NAT operation needs to be...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png']
1,"Logstash JDBC input, filter event timestamp di...",\n \nI am trying to change the ...,"['elasticsearch', 'logstash', 'elastic-stack',...",https://stackoverflow.com/questions/33787795/l...,0,2015-11-18 18:38:40Z,3,"['could you suggest how I could fix?', 'Your d...",[{'body': '\nOK I finally figure how to get th...,\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


## Processed

In [18]:
device=device[['title','body','selected_answer','images']]

In [19]:
device.head(2)

,title,body,selected_answer,images
0,What is the purpose of the INPUT chain in the ...,\n \nWhat is the purpose of the...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png']
1,"Logstash JDBC input, filter event timestamp di...",\n \nI am trying to change the ...,\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


In [20]:
device['context'] = "title: " + device['title'].astype(str) + "\nbody: " + device['body'].astype(str).str.strip()

In [21]:
device_processed=pd.DataFrame(
    {
        'context':device['context'],
        'gt':device['selected_answer'],
        'img':device['images']
    }
)

In [22]:
device_processed.head(2)

,context,gt,img
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png']
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


# Prompt Setup

## Cloud

### Zero Shot

In [23]:
zero_shot = """
You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
{}

Answer:
"""


In [24]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['context'])

cloud_processed['zero-shot-question'] = cloud_processed.apply(generate_zero_shot_prompt, axis=1)

In [25]:
print(cloud_processed['zero-shot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Answer:



### CoT

In [26]:
cot = '''
You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
{}

Lets think step by step:
Answer:
'''

In [27]:
def generate_cot_prompt(row):
    return cot.format(row['context'])

cloud_processed['cot-question'] = cloud_processed.apply(generate_cot_prompt, axis=1)


In [28]:
print(cloud_processed['cot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Lets think step by step:
Answer:



## Device

### Zero Shot

In [29]:
zero_shot = '''
You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
{}

Answer:

'''

In [30]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['context'])

device_processed['zero-shot-question'] =device_processed.apply(generate_zero_shot_prompt, axis=1)


In [31]:
print(device_processed['zero-shot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
title: What is the purpose of the INPUT chain in the nat table?
body: What is the purpose of the INPUT chain in the nat table ?

The PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".

The nat:INPUT and nat:PREROUTING chains seem redundant.

What would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?

Answer:




### CoT

In [32]:
cot = '''
You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
{}

Lets think step by step:
Answer:
'''

In [33]:
def generate_cot_prompt(row):
    return cot.format(row['context'])

device_processed['cot-question'] = device_processed.apply(generate_cot_prompt, axis=1)

In [34]:
print(device_processed['cot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
title: What is the purpose of the INPUT chain in the nat table?
body: What is the purpose of the INPUT chain in the nat table ?

The PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".

The nat:INPUT and nat:PREROUTING chains seem redundant.

What would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?

Lets think step by step:
Answer:



## Prompt Length

In [35]:
# extract prompt length
def prompt_length(text, add_special_tokens=True):
    if pd.isnull(text) or not isinstance(text, str) or not text.strip():
        return 0
    temp_inputs = processor.tokenizer(text, return_tensors="pt", add_special_tokens=add_special_tokens)
    return temp_inputs.input_ids.shape[1]


In [36]:
# Prompt Length -> Cloud Logs
cloud_processed['zero-shot-question-length'] = cloud_processed['zero-shot-question'].apply(prompt_length)
cloud_processed['cot-question-length'] = cloud_processed['cot-question'].apply(prompt_length)

In [37]:
# Prompt Length -> Device Logs
device_processed['zero-shot-question-length'] = device_processed['zero-shot-question'].apply(prompt_length)
device_processed['cot-question-length'] = device_processed['cot-question'].apply(prompt_length)

# Message Setup ( Prompt + Images )

In [38]:
import ast

def setup_messages(dataset, prompt_column="", image_column="img"):
    messages = []

    for _, row in dataset.iterrows():
        imgs = row.get(image_column, [])
        if isinstance(imgs, str):
            try:
                imgs = ast.literal_eval(imgs)
            except Exception:
                imgs = []

        if imgs:
            # Build structured message with image placeholders and direct question
            content = [{"type": "image", "image": img} for img in imgs if img]
            content.append({
                "type": "text",
                "text": row.get(prompt_column, "")
            })

            messages.append({
                "role": "user",
                "qa": content
            })

    return messages


In [39]:
messages = setup_messages(device_processed,prompt_column="cot-question")

In [40]:
messages[1]

{'role': 'user',
 'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/bsoVL.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/wM2pW.png'},
  {'type': 'text',
   'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: Logstash JDBC input, filter event timestamp different to @timestamp\nbody: I am trying to change the @timestamp to match my timestamp, this is the time my event occurred at not when logstash time stamped it.\n\nhere is my conf file\n\ninput {\n jdbc {\n   jdbc_driver_library => "C:/logstash/lib/mysql-connector-java-5.1.37-bin.jar"\n   jdbc_driver_class => "com.mysql.jdbc.Driver"\n   jdbc_connection_string => "jdbc:mysql://127.0.0.1:3306/transport"\n   jdbc_user => "root"\n   jdbc_password => ""\n   schedule => "* * * * *"\n   stat

# Generate Response

## Setup Model

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)  # move model to device once
model.eval()

LlavaForConditionalGeneration(
  (model): LlavaModel(
    (vision_tower): CLIPVisionModel(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(577, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
       

In [42]:
import math
from PIL import Image

MAX_LONG_EDGE = 336   
PATCH_SIZE    = 14
MIN_EDGE      = 28

def img_resize(img: Image.Image) -> Image.Image:
    w, h = img.size

    # Downscale if necessary
    scale = min(1.0, MAX_LONG_EDGE / max(w, h))
    w, h = int(round(w * scale)), int(round(h * scale))

    # Ensure minimum size
    w, h = max(w, MIN_EDGE), max(h, MIN_EDGE)

    # Round to nearest multiple of patch size
    w = math.ceil(w / PATCH_SIZE) * PATCH_SIZE
    h = math.ceil(h / PATCH_SIZE) * PATCH_SIZE

    resized_img = img.resize((w, h), Image.Resampling.LANCZOS)
    return resized_img


In [50]:
import torch
import gc
import traceback
import requests
torch.set_grad_enabled(False)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
torch.set_grad_enabled(False)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Load and prepare image(s) from message
def process_vision_info(messages):
    images = []
    for msg in messages:
        for item in msg.get("content", []):
            if item["type"] == "image":
                img = item["image"]
                if isinstance(img, str):
                    img = Image.open(requests.get(img, stream=True).raw).convert("RGB")
                images.append(img_resize(img))
    return images

# Process message with debug info
def process_message(msg: dict, idx: int):
    try:
        prompt_str = processor.apply_chat_template(
            [msg], tokenize=False, add_generation_prompt=True
        )

        imgs = process_vision_info([msg])
        if not imgs:
            raise ValueError("No images found in message.")

        inputs = processor(
            text=[prompt_str],
            images=imgs,
            return_tensors="pt"
        ).to(model.device)

        gen_ids = model.generate(
            **inputs,
            top_p=0.89,
            temperature=1.8,
            do_sample=True,
            max_new_tokens=768,
            use_cache=True
        )

        reply_ids = gen_ids[:, inputs.input_ids.shape[-1]:]
        answer = processor.batch_decode(
            reply_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        return {"idx": idx, "response": answer}

    except Exception as e:
        return {
            "idx": idx,
            "response": f"[ERROR] {e}",
            "traceback": traceback.format_exc()
        }

    finally:
        for name in ("inputs", "gen_ids"):
            if name in locals():
                del locals()[name]
        torch.cuda.empty_cache()
        gc.collect()


## Testing Samples

In [51]:
#sample=cloud_processed[:5]
sample=device_processed[:10]

In [45]:
sample.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png'],\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,175,181
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,730,736


In [104]:
# Setup messages for model
sample_messages = setup_messages(sample,prompt_column="zero-shot-question")
#sample_messages = setup_messages(sample,prompt_column="cot-question")

In [105]:
sample_messages[:2]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'},
   {'type': 'text',
    'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nAnswer:\n\n'}]},
 {'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
   {'type': 'image', 'image': 'https://i.sstatic.net/bsoVL.png'},
   {'ty

### Get response for Sample  -> Test

In [106]:
# Output setup
sample_results = []
output_path = "sample_results_full.jsonl"

# Ensure column exists for generated responses
sample["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

# Process each message
for row_idx, msg in enumerate(tqdm(sample_messages, desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    sample.at[row_idx, "generated_response"] = response_text

    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)

    # Save row result
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    sample_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


/tmp/ipykernel_436/2944900821.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample["generated_response"] = None
Processing Messages:  10%|█         | 1/10 [00:07<01:08,  7.61s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'}, {'type': 'text', 'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nAnswer:\n\n'}]

##Sample Response:##

The purpose of the INPUT chain in the nat table is for translating the source addresses (SA) of outgoing packets. The nat:INPUT chain is used when the packet move

Processing Messages: 100%|██████████| 10/10 [00:34<00:00,  3.45s/it]


In [107]:
sample_results

[{'row': 0,
  'response': 'The purpose of the INPUT chain in the nat table is for translating the source addresses (SA) of outgoing packets. The nat:INPUT chain is used when the packet moves from the PREROUTING stage to the OUTPUT stage. While the nat:PREROUTING chain translates the destination addresses (DNAT) of incoming packets, the nat:INPUT chain translates the source addresses (SA) of outgoing packets, allowing for traffic that bypasses the PREROUTING table, such as traffic moving directly to an internal server.',
  'image_urls': ['https://i.sstatic.net/NUh2k.png']},
 {'row': 1,
  'response': 'Yes, it seems that there could be a problem related to the timestamp handling since it comes from a MySQL database with different time handling as compared to the Elasticsearch timestamp. One possible issue is the handling of timezone information in the data. In MySQL, timestamps do not include timezone information, whereas Elasticsearch expects the timestamp to be timezone-adjusted when it

In [92]:
sample_results

[{'row': 0,
  'response': 'The INPUT chain is typically used in the Network Address Translation (NAT) table to process incoming traffic. It is likely designed to allow control over the handling of traffic that is not a result of translation. It could be used to modify routing table entries based on the traffic properties or could be used to manage access and authentication for services that use translating routing table entries. The exact purpose depends on the infrastructure and specific requirements of the network, which could not be inferred from a visual representation like the one on display.',
  'image_urls': ['https://i.sstatic.net/NUh2k.png']},
 {'row': 1,
  'response': "The issue appears to be related to the difference between the original `@timestamp` value and the value that logstash has assigned to it, potentially due to MySQL's datestamp behavior and conversion to a local timestamp in logstash. To resolve the problem, you can create an input filter that formats the origina

## Cloud

In [ ]:
cloud_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,121,127
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,392,398


### Zero-shot

In [ ]:
# Setup messages for mode
cloud_messages = setup_messages(cloud_processed,prompt_column="zero-shot-question")

In [ ]:
cloud_messages[0:1]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'},
   {'type': 'text',
    'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and take the provided images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nAnswer:\n'}]}]

#### Get response

In [ ]:
# Output setup
cloud_logs_zero_shot_results = []
output_path = "vlm_cloud_llava1_5_vl_7b_zero_shot_results.jsonl"

# Ensure column exists for generated responses
cloud_processed["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

# Process each message
for row_idx, msg in enumerate(tqdm(cloud_messages, desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    device_processed.at[row_idx, "generated_response"] = response_text

    #Sample Response
    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)

    # Save row result
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    cloud_logs_zero_shot_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


In [ ]:
cloud_processed['zero-shot-prompt'] = setup_messages(cloud_processed, prompt_column="zero-shot-question")


In [ ]:
cloud_processed.head(2)

In [ ]:
cloud_processed.to_excel("../responses/cloud/vlm_response_cloud_llava1_5_vl_7b_zero_shot.xlsx", index=False)

### CoT

In [ ]:
# Setup messages for mode
cloud_messages = setup_messages(cloud_processed,prompt_column="cot-question")

In [ ]:
cloud_messages[0:1]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'},
   {'type': 'text',
    'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and take the provided images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nLets think step by step:\nAnswer:\n'}]}]

#### Get response

In [ ]:
# Output setup
cloud_logs_cot_results = []
output_path = "vlm_cloud_llava1_5_vl_7b_cot_results.jsonl"

# Ensure column exists for generated responses
cloud_processed["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

# Process each message
for row_idx, msg in enumerate(tqdm(cloud_messages, desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    device_processed.at[row_idx, "generated_response"] = response_text

    #Sample Response
    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)

    # Save row result
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    cloud_logs_cot_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


In [ ]:
cloud_processed['cot-prompt'] = setup_messages(cloud_processed, prompt_column="cot-question")


In [ ]:
cloud_processed.head(2)

In [ ]:
cloud_processed.to_excel("../responses/cloud/vlm_response_cloud_llava1_5_vl_7b_cot.xlsx", index=False)

## Device

In [52]:
device_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,generated_response
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png'],\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,175,181,The purpose of the INPUT chain in the nat tabl...
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,730,736,"Yes, it seems that there could be a problem re..."


### Zero-Shot

In [53]:
# Setup messages for mode
device_messages = setup_messages(device_processed,prompt_column="zero-shot-question")

In [54]:
device_messages[:2]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'},
   {'type': 'text',
    'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nAnswer:\n\n'}]},
 {'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
   {'type': 'image', 'image': 'https://i.sstatic.net/bsoVL.png'},
   {'ty

#### Get response

In [55]:
# Output setup
device_logs_zero_shot_results = []
output_path = "vlm_device_llava1_5_vl_7b_zero_shot_results.jsonl"

# Ensure column exists for generated responses
device_processed["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

# Process each message
for row_idx, msg in enumerate(tqdm(device_messages, desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    device_processed.at[row_idx, "generated_response"] = response_text

    #Sample Response
    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)


    # Save row result
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    device_logs_zero_shot_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


Processing Messages:   0%|          | 1/993 [00:07<2:11:46,  7.97s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'}, {'type': 'text', 'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nAnswer:\n\n'}]

##Sample Response:##

There are instructions that can only be executed through the "nat" chain named `ip\_t` and `input chain`, which serves as a forwarders route between the `destinat

Processing Messages:  16%|█▋        | 162/993 [25:40<1:08:19,  4.93s/it]/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Processing Messages: 100%|██████████| 993/993 [2:32:00<00:00,  9.18s/it]  


In [56]:
device_logs_zero_shot_results[:2]

[{'row': 0,
  'response': 'There are instructions that can only be executed through the "nat" chain named `ip\\_t` and `input chain`, which serves as a forwarders route between the `destination route subgraph`, where `input chain` handles filtering the received `packet routes`, `destination ip route table `that gets outputed. They also contain some code regarding outputting table, and there\'s a connection to it. Additionally, the images feature "nat", "ip\\_t", and "outcoming link" representations, as well as others like "routing", "inputs" and others.',
  'image_urls': ['https://i.sstatic.net/NUh2k.png']},
 {'row': 1,
  'response': 'It looks like you want the Elasticsearch plugin to read and sort data based on a different timezone or timestamp field. Unfortunately, changing the timestamp on filter side seems to create another issue - `_dateparsefailure`. A potential solution might be applying the necessary adjustment on input plugin level for the MySQL database field. Check the MySQL

In [57]:
device_processed['zero-shot-prompt'] = setup_messages(device_processed, prompt_column="zero-shot-question")

In [58]:
device_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,generated_response,zero-shot-prompt
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png'],\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,175,181,There are instructions that can only be execut...,"{'role': 'user', 'qa': [{'type': 'image', 'ima..."
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,730,736,It looks like you want the Elasticsearch plugi...,"{'role': 'user', 'qa': [{'type': 'image', 'ima..."


In [59]:
device_processed.to_excel("../responses/device/vlm_response_device_llava1_5_vl_7b_zero_shot.xlsx", index=False)

### CoT

In [60]:
# Setup messages for mode
device_messages = setup_messages(device_processed,prompt_column="cot-question")

In [61]:
device_messages[:2]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'},
   {'type': 'text',
    'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nLets think step by step:\nAnswer:\n'}]},
 {'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
   {'type': 'image', 'image': 'https://i.sstatic.ne

#### Get response

In [62]:
# Output setup
device_logs_cot_results = []
output_path = "vlm_device_llava1_5_vl_7b_cot_results.jsonl"

# Ensure column exists for generated responses
device_processed["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

# Process each message
for row_idx, msg in enumerate(tqdm(device_messages, desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    device_processed.at[row_idx, "generated_response"] = response_text

    #Sample Response
    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)


    # Save row result
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    device_logs_cot_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


Processing Messages:   0%|          | 1/993 [00:17<4:50:14, 17.56s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'}, {'type': 'text', 'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nLets think step by step:\nAnswer:\n'}]

##Sample Response:##

The `INPUT` chain in the `nat` table has a purpose during a part of packet traffic, where traffic first entered this `INPUT` chain, likely d

Processing Messages:  16%|█▋        | 162/993 [31:30<2:42:39, 11.74s/it]/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Processing Messages: 100%|██████████| 993/993 [2:54:13<00:00, 10.53s/it]  


In [63]:
device_logs_cot_results[:2]

[{'row': 0,
  'response': 'The `INPUT` chain in the `nat` table has a purpose during a part of packet traffic, where traffic first entered this `INPUT` chain, likely due to some type of filtering to enter this `INPUT` chain being executed from the upper `nat` chains or top of the switch stack such a firewall before entering router, the filter for instance will add an output direct (OD) command to this chain on top if the input direct chain was in use on top. No chain above it would remove a route for traffic and if this happens after `input` chain rules, the next chain would not remove the route; this helps that the route persistence can occur here or you could do the final route check of that some filter/router before going in the `INPUT` chain. Additionally, rules may change a table element or it may only apply rules to this table element in the case that the INPUT chain is not present then these rules don have an element to rule over as there be rules above for a same destination IP

In [64]:
device_processed['cot-prompt'] = setup_messages(device_processed, prompt_column="cot-question")

In [65]:
device_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,generated_response,zero-shot-prompt,cot-prompt
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png'],\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,175,181,The `INPUT` chain in the `nat` table has a pur...,"{'role': 'user', 'qa': [{'type': 'image', 'ima...","{'role': 'user', 'qa': [{'type': 'image', 'ima..."
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,730,736,No answer,"{'role': 'user', 'qa': [{'type': 'image', 'ima...","{'role': 'user', 'qa': [{'type': 'image', 'ima..."


In [66]:
device_processed.to_excel("../responses/device/vlm_response_device_llava1_5_vl_7b_cot.xlsx", index=False)